# Context Managers

The `with` statement is one of Python's most underrated features. It guarantees that setup and teardown code always runs, even when exceptions occur, without writing a single `try/finally` block. Custom context managers let you wrap any resource acquisition pattern in a clean, reusable interface.

**What's inside:** `with` mechanics, writing class-based context managers with `__enter__`/`__exit__`, `contextlib.contextmanager` for generator-based managers, `contextlib.suppress`, and `contextlib.ExitStack` for dynamic composition.

**Learn more:** [contextlib](https://docs.python.org/3/library/contextlib.html) · [with statement](https://docs.python.org/3/reference/compound_stmts.html#the-with-statement)

## 1. The `with` Statement

`with` calls `__enter__` on entry and `__exit__` on exit, guaranteed even if an exception is raised inside the block.

### 1.1 File I/O: the canonical example

In [ ]:
import tempfile, os

# without with: file may not be closed if an exception occurs
f = open('/dev/null')
f.read()
f.close()

# with with: file is always closed, even on error
with open('/dev/null') as f:
    data = f.read()

# f is closed here (attempting f.read() would raise ValueError)
print(f.closed)

### 1.2 Multiple context managers in one line

In [ ]:
import tempfile

# open two temp files at once (both are guaranteed to close)
with tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False) as src, \
     tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False) as dst:
    src.write('hello')
    dst.write('world')
    print(f'src: {src.name}')
    print(f'dst: {dst.name}')

# both files are closed here
print(f'src closed: {src.closed}, dst closed: {dst.closed}')

## 2. Class-Based Context Managers

Implement `__enter__` (setup, returns the managed resource) and `__exit__` (teardown, receives exception info).

In [ ]:
import time

class Timer:
    def __enter__(self):
        self.start = time.perf_counter()
        return self          # this becomes the `as` variable

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.elapsed = time.perf_counter() - self.start
        # return False (or None) to let exceptions propagate
        return False

with Timer() as t:
    total = sum(range(1_000_000))

print(f'elapsed: {t.elapsed:.4f}s')
print(f'result:  {total:,}')

In [ ]:
# __exit__ can suppress exceptions by returning True
class Suppressor:
    def __init__(self, *exception_types):
        self.exception_types = exception_types

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        if exc_type and issubclass(exc_type, self.exception_types):
            print(f'suppressed {exc_type.__name__}: {exc_val}')
            return True   # swallow the exception
        return False

with Suppressor(ZeroDivisionError, ValueError):
    x = 1 / 0            # suppressed (no crash)

print('execution continued normally')

## 3. Generator-Based Context Managers

`contextlib.contextmanager` turns a generator function into a context manager. Everything before `yield` is `__enter__`; everything after is `__exit__`.

In [ ]:
from contextlib import contextmanager
import time

@contextmanager
def timer(label=''):
    start = time.perf_counter()
    try:
        yield                          # the with-block runs here
    finally:
        elapsed = time.perf_counter() - start
        print(f'{label}: {elapsed:.4f}s')

with timer('sum of squares'):
    result = sum(x**2 for x in range(1_000_000))
    
print(f'result: {result:,}')

In [ ]:
# practical: temporarily change working directory
import os

@contextmanager
def working_directory(path):
    original = os.getcwd()
    os.chdir(path)
    try:
        yield
    finally:
        os.chdir(original)

print(f'before: {os.getcwd()}')
with working_directory('/tmp'):
    print(f'inside: {os.getcwd()}')
print(f'after:  {os.getcwd()}')

In [ ]:
# yield the resource so `as` can bind it
@contextmanager
def managed_list():
    items = []
    yield items          # the list is passed to `as`
    print(f'final list: {items}')

with managed_list() as lst:
    lst.append(1)
    lst.append(2)
    lst.append(3)

## 4. contextlib Utilities

`contextlib` ships several ready-made context managers for common patterns.

### 4.1 suppress: silence specific exceptions

In [ ]:
from contextlib import suppress
import os

# without suppress
try:
    os.remove('nonexistent_file.txt')
except FileNotFoundError:
    pass

# with suppress (same effect, less ceremony)
with suppress(FileNotFoundError):
    os.remove('nonexistent_file.txt')

print('done, no crash')

### 4.2 redirect_stdout: capture or redirect output

In [ ]:
from contextlib import redirect_stdout
import io

buffer = io.StringIO()
with redirect_stdout(buffer):
    print('this goes to the buffer, not the terminal')
    print('so does this')

captured = buffer.getvalue()
print(f'captured {len(captured.splitlines())} lines:')
print(captured)

### 4.3 ExitStack: compose context managers dynamically

In [ ]:
from contextlib import ExitStack
import tempfile

# open a variable number of files (impossible with a fixed `with` line)
filenames = [tempfile.mktemp() for _ in range(3)]

with ExitStack() as stack:
    files = [stack.enter_context(open(f, 'w')) for f in filenames]
    for i, f in enumerate(files):
        f.write(f'file {i}')
    print(f'opened {len(files)} files inside the block')

# all files are closed here
print(f'all closed: {all(f.closed for f in files)}')

## 5. Real-World Patterns

In [ ]:
# database transaction: commit on success, rollback on failure
from contextlib import contextmanager

class FakeDB:
    def execute(self, sql): print(f'  execute: {sql}')
    def commit(self):       print('  commit')
    def rollback(self):     print('  rollback')

@contextmanager
def transaction(db):
    try:
        yield db
        db.commit()
    except Exception as e:
        db.rollback()
        raise

db = FakeDB()
with transaction(db) as conn:
    conn.execute('INSERT INTO users VALUES (1, "Alice")')
    conn.execute('INSERT INTO orders VALUES (1, 1, 99.99)')

In [ ]:
# temporarily patch an object attribute
from contextlib import contextmanager

@contextmanager
def patched(obj, attr, value):
    original = getattr(obj, attr)
    setattr(obj, attr, value)
    try:
        yield
    finally:
        setattr(obj, attr, original)

class Config:
    debug = False

cfg = Config()
print(f'before: debug={cfg.debug}')
with patched(cfg, 'debug', True):
    print(f'inside: debug={cfg.debug}')
print(f'after:  debug={cfg.debug}')